# i.MX 93 AHAB with U-BOOT

This notebook describes how to build a bootable AHAB image with ELE firmware and U-BOOT bootloader, load it using the NXPUUU app and how to use NXPELE app.


## 1. Prerequisites

- SPSDK is needed with examples extension. `pip install spsdk[examples]` (Please refer to the installation documentation.)
- This demo was tested with i.MX93 EVK board

### 1.1 Images preparation

- to create resulting binary containing AHAB containers, we need to prepare the binaries
- in this section we reproduce the process which is done by the `imx-mkimage` tool as flash singleboot binary
- Obtain all the necessary binaries (lpddr4 firmware, u-boot, bl31, ELE firmware) and put them into inputs directory

### 1.2 U-Boot

[Read the U-Boot documentation to understand the U-Boot build process](https://docs.u-boot.org/en/latest/board/nxp/imx93_11x11_evk.html#quick-start)
In order to use the nxpele app, U-Boot must be built with AHAB support. CONFIG_AHAB_BOOT=y
If you want to use the nxpele over fastboot, also multiplexing of console output to fastboot must be enabled by setting CONFIG_CONSOLE_MUX=y.

### 1.3 Requirements

- lpddr4 firmware files
- u-boot binary (u-boot SPL and u-boot) built with AHAB support.
- bl31.bin binary (ARM Trusted Firmware)
- ELE firmware binary AHAB 
- i.MX93 EVK board


In [1]:
# This env variable sets colored logger output to STDOUT
# Execute this cell to enable execution of the ! line magic
%env JUPYTER_SPSDK=1
%alias execute echo %l && %l
%alias_magic ! execute

env: JUPYTER_SPSDK=1
Created `%!` as an alias for `%execute`.


## 2. Bootable image

The imx-boot image consists of two container sets. The first container set is loaded to OCRAM using the SDPS protocol supported by ROM. This set contains the U-Boot SPL binary, ELE firmware, and training data for the DDR controller. The second container set contains the full U-Boot and ATF image and is loaded to DDR by U-Boot SPL.


### 2.1 Bootable image template

You can generate the template using the `nxpimage bootable-image get-templates` command. This will generate all possible templates for building the imx-boot image. You can also use the `--template` parameter to generate only a specific template folder (possible options can be listed with the command `nxpimage bootable-image list-templates`).

The following command generates the template folder:

```bash
nxpimage bootable-image get-templates -f mimx9352 -t imx_boot_flash_singleboot -o bootable_image_templates
```

### 2.2 Exporting of the imx-boot image

The imx-boot image can be exported using the `nxpimage bootable-image export` command: 

```bash
nxpimage bootable-image export -c bootable_image_templates/bootable_image.yaml 
```



## 3. Exporting the image

In [2]:
# Let's create a template folder for our EVK board
VERBOSITY = "-v"
FAMILY = "mimx9352"
TEMPLATE_FOLDER = "workspace/bootable_image_template"
BOOTABLE_IMAGE_CONFIG = TEMPLATE_FOLDER + "/bootable_image.yaml"
BOOTABLE_IMAGE = TEMPLATE_FOLDER + "/bootable_image.bin"
%! nxpimage $VERBOSITY bootable-image get-templates -f $FAMILY -t imx_boot_flash_singleboot -b imx93-9x9-lpddr4-qsb -o $TEMPLATE_FOLDER --force

nxpimage -v bootable-image get-templates -f mimx9352 -t imx_boot_flash_singleboot -b imx93-9x9-lpddr4-qsb -o workspace/bootable_image_template --force 
Creating input directory: inputs

Template 'imx_boot_flash_singleboot' generated successfully!
Description: Complete imx-bootloader image
Board: imx93-9x9-lpddr4-qsb
Input directory: inputs

Generated files in workspace/bootable_image_template:
  - spl.yaml
  - uboot.yaml
  - bootable_image.yaml

To create the bootable image, run:
nxpimage bootable-image export -c workspace/bootable_image_template/bootable_image.yaml


In [3]:
# Now move your binaries to the generated input folder and check if the names in the configs match the file names
%! nxpimage bootable-image export -c $BOOTABLE_IMAGE_CONFIG

nxpimage bootable-image export -c workspace/bootable_image_template/bootable_image.yaml 
Success. (Bootable Image: workspace/bootable_image_template/bootable_image.bin created) 


## 4. Image download
First we put the `iMX93` into serial downloader mode for Cortex-A (`0011`), and use `uuu` to upload the `flash.bin` containing U-Boot and other firmware. 

The picture below shows the desired DIP switch configuration for flashing Cortex-A:

![Download Mode Cortex-A](img/serial_downloader.jpg)

In [4]:
%! nxpuuu $VERBOSITY write -f $FAMILY -b emmc $BOOTABLE_IMAGE

nxpuuu -v write -f mimx9352 -b emmc workspace/bootable_image_template/bootable_image.bin 
SDPS: boot -f workspace/bootable_image_template/bootable_image.bin


Success


## 5. NXPELE

NXPELE is a tool that can communicate with the EdgeLock enclave. It might be used for example for writing and reading the fuses and keyblob generation.

To communicate with the `iMX93`, we need to set it into eMMC boot for Cortex-A (`0000`)

The picture below shows the desired DIP switch configuration for booting from Cortex-A:

![Boot from Cortex-A](img/emmc_boot.jpg)

In [5]:
%! nxpele -f $FAMILY get-info

nxpele -f mimx9352 get-info 
ELE get info ends successfully:
Command:              0xda
Version:              2
Length:               160
SoC ID:               MX93 - 0x9300
SoC version:          A100
Life Cycle:           OEM_OPEN - 0x0010
SSSM state:           4
Attest API version:   2
UUID:                 da9634a77b614cd7a3dc5c147ffc940f
SHA256 ROM PATCH:     e9b0338e5f4a0a92025f764c5eeae2d26be1211c77ee51e49a9ee36a7185d587
SHA256 FW:            4f490c747544c3411ae3135c7e5ce5e02d078b47af0eba5b88c01a980c18bff6
Advanced information:
  OEM SRKH:           e7799ca6358d8b57b912f55d65ee135c2f22062fab9987ea4ba3252a9a69b33a
  IMEM state:         The IMEM is fully loaded and all ELE functionality can be used - 0xCA
  CSAL state:         EdgeLock secure enclave random context initialization succeed - 0x02
  TRNG state:         TRNG entropy is valid and ready to be read - 0x03

